In [1]:
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit import QuantumCircuit
from qiskit.circuit.library import n_local
from qiskit_algorithms.optimizers import COBYLA
import numpy as np

In [2]:
#Building the frustrated Ising Hamiltonian with transverse field
N = 3; #Number of spins
coeffs = [0.5,0.5]; #[J,h]
H = ( SparsePauliOp.from_list([("ZZI", coeffs[0])]) +
          SparsePauliOp.from_list([("ZIZ", coeffs[0])]) +
          SparsePauliOp.from_list([("IZZ", coeffs[0])]) +
          SparsePauliOp.from_list([("XII", coeffs[1])]) +
          SparsePauliOp.from_list([("IXI", coeffs[1])]) + 
          SparsePauliOp.from_list([("IIX", coeffs[1])]) )

In [3]:
#Ry-CZ Ansatz for the trial wavefunction 

p = N*(3 + 1); #Total number of parameters with reps = 3;
ansatz = n_local(N, rotation_blocks = 'ry', entanglement_blocks = 'cz', reps=3)


#Defining the expectation value
def expectation_value(params):
    bound_qc = ansatz.assign_parameters(params)
    psi = Statevector(bound_qc)
    return psi.expectation_value(H).real

#Finding ground state energy using COBYLA
optimizer = COBYLA(maxiter=1000, tol=1e-6)
result = optimizer.minimize(fun = expectation_value, x0 = np.array([1/(np.random.randint(1,10)) for _ in range(p)]))

print(f"Ground state energy: {result.fun:.5f} Hartree")

Ground state energy: -1.73205 Hartree


In [4]:
#Extracting the ground state

#Binding the optimal parameters
optimal_params = result.x
bound_qc = ansatz.assign_parameters(
    {p: v for p, v in zip(ansatz.parameters, optimal_params)}
)

#Statevector
psi = Statevector(bound_qc)
ground_state = psi.data;

print("\nBasis state   Amplitude              Probability")
for idx, amp in enumerate(ground_state):
    prob = abs(amp)**2
    if prob > 0.01:  # only show components > 1%
        print(f"|{idx:03b}>       {amp.real:+.4f}{amp.imag:+.4f}j     {prob:.4f}")


Basis state   Amplitude              Probability
|000>       -0.1830+0.0000j     0.0335
|001>       +0.3943+0.0000j     0.1555
|010>       +0.3943+0.0000j     0.1555
|011>       -0.3943+0.0000j     0.1555
|100>       +0.3943+0.0000j     0.1555
|101>       -0.3943+0.0000j     0.1555
|110>       -0.3943+0.0000j     0.1555
|111>       +0.1830+0.0000j     0.0335


In [5]:
#Verifying against exact diagonalization

# Exact ground state
eigenvalues, eigenvectors = np.linalg.eigh(H.simplify().to_matrix())
ed_ground_state = eigenvectors[:, 0]

# Compare
overlap = abs(np.dot(ground_state.conj(), ed_ground_state))**2
print(f"\nOverlap with exact ground state: {overlap:.6f}")


Overlap with exact ground state: 1.000000
